# Indonesian Text Summarization Pipeline

End-to-end pipeline for summarizing Indonesian public complaint texts into short formal summaries.

**Model:** `csebuetnlp/mT5_multilingual_XLSum` (multilingual summarization model trained on XLSum dataset)

**Pipeline Steps:**
1. Install dependencies
2. Load dataset
3. Text preprocessing
4. Load model and tokenizer
5. Run summarization
6. Print results

## 1. Install Dependencies

In [1]:
!pip install transformers sentencepiece protobuf torch --quiet

## 2. Load Dataset

Using a mock dataset of Indonesian public complaint texts written in informal language.

In [2]:
import pandas as pd

complaints = [
    "Jalan di kampung kami rusak parah sudah lama dan kalau hujan banjir. "
    "Warga udah lapor ke kelurahan tapi gak ada tanggapan sama sekali. "
    "Anak-anak sekolah jadi susah lewat dan sering jatuh kena lubang.",

    "Lampu jalan di perumahan kami mati udah 3 bulan lebih. "
    "Malam hari gelap banget dan rawan kejahatan. "
    "Sudah lapor ke PLN dan RT tapi belum ada perbaikan juga.",

    "Sampah di pasar deket rumah numpuk tinggi dan bau banget. "
    "Truk sampah udah seminggu gak dateng. Banyak lalat dan tikus berkeliaran. "
    "Warga sekitar takut kena penyakit apalagi anak-anak kecil.",
]

df = pd.DataFrame({"complaint_text": complaints})
print(f"Total complaints: {len(df)}")
df

Total complaints: 3


,complaint_text
0,Jalan di kampung kami rusak parah sudah lama d...
1,Lampu jalan di perumahan kami mati udah 3 bula...
2,Sampah di pasar deket rumah numpuk tinggi dan ...


## 3. Text Preprocessing

Clean the raw complaint texts before feeding them to the model:
- Normalize whitespace
- Remove non-alphanumeric characters (keep Indonesian letters)
- Convert to lowercase

In [3]:
import re


def preprocess_text(text: str) -> str:
    """Clean and normalize Indonesian text for summarization."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s.,\-]", "", text)  # keep letters, digits, basic punctuation
    text = re.sub(r"\s+", " ", text).strip()       # collapse multiple spaces
    return text


# Apply preprocessing
df["cleaned_text"] = df["complaint_text"].apply(preprocess_text)

# Show before vs after
for i, row in df.iterrows():
    print(f"--- Complaint {i+1} ---")
    print(f"Original : {row['complaint_text'][:80]}...")
    print(f"Cleaned  : {row['cleaned_text'][:80]}...")
    print()

--- Complaint 1 ---
Original : Jalan di kampung kami rusak parah sudah lama dan kalau hujan banjir. Warga udah ...
Cleaned  : jalan di kampung kami rusak parah sudah lama dan kalau hujan banjir. warga udah ...

--- Complaint 2 ---
Original : Lampu jalan di perumahan kami mati udah 3 bulan lebih. Malam hari gelap banget d...
Cleaned  : lampu jalan di perumahan kami mati udah 3 bulan lebih. malam hari gelap banget d...

--- Complaint 3 ---
Original : Sampah di pasar deket rumah numpuk tinggi dan bau banget. Truk sampah udah semin...
Cleaned  : sampah di pasar deket rumah numpuk tinggi dan bau banget. truk sampah udah semin...



## 4. Load Model and Tokenizer

Load `csebuetnlp/mT5_multilingual_XLSum`, a multilingual summarization model pretrained on the XLSum dataset that supports Indonesian.

> First run will download the model. Subsequent runs use the cached version.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Can't load the model for 'csebuetnlp/mT5_multilingual_XLSum'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'csebuetnlp/mT5_multilingual_XLSum' is the correct path to a directory containing a file named pytorch_model.bin.

## 5. Define Summarization Function

Reusable function that tokenizes input, generates summary tokens with beam search, and decodes back to text.

In [ ]:
WHITESPACE_HANDLER = lambda k: re.sub(r"\s+", " ", re.sub(r"\n+", " ", k)).strip()


def summarize_texts(texts: list[str], max_length: int = 84, min_length: int = 10) -> list[str]:
    """
    Summarize a list of texts using the loaded mT5 model.

    Args:
        texts: List of input texts to summarize.
        max_length: Maximum number of tokens in the generated summary.
        min_length: Minimum number of tokens in the generated summary.

    Returns:
        List of generated summary strings.
    """
    summaries = []
    for text in texts:
        # Tokenize input with truncation for long texts
        inputs = tokenizer(
            [WHITESPACE_HANDLER(text)],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=512,
        )

        # Generate summary tokens
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            min_length=min_length,
            no_repeat_ngram_size=3,
            num_beams=5,
        )

        # Decode tokens back to text
        summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        summaries.append(summary)

    return summaries


print("Summarization function ready.")

## 6. Run Inference and Print Results

Generate summaries for all complaint texts and display the results side by side.

In [ ]:
# Run summarization on the cleaned texts
summaries = summarize_texts(df["cleaned_text"].tolist())

# Store summaries back into the dataframe
df["summary"] = summaries

# Print results
print("=" * 70)
print("INDONESIAN COMPLAINT TEXT SUMMARIZATION RESULTS")
print("=" * 70)

for i, row in df.iterrows():
    print(f"\n[Complaint {i+1}]")
    print(f"Input   : {row['complaint_text']}")
    print(f"Summary : {row['summary']}")
    print("-" * 70)

## 7. Summary Table

View all results in a clean table format.

In [ ]:
# Display final dataframe with original text and generated summary
df[["complaint_text", "summary"]]